In [ ]:
import pandas as pd
import numpy as np

# analysis window
START_DATE = pd.Timestamp("2023-12-29")
END_DATE   = pd.Timestamp("2025-12-31")


In [ ]:
def load_sec_px(path: str) -> pd.DataFrame:
    px = pd.read_csv(
        path,
        parse_dates=["date"],
        dayfirst=False
    )
    px = px.set_index("date").sort_index()
    px = px.apply(pd.to_numeric, errors="coerce")
    return px
sec_px = load_sec_px("sec_px.csv")

def restrict_window(px: pd.DataFrame,
                    start: pd.Timestamp,
                    end: pd.Timestamp) -> pd.DataFrame:
    return px.loc[(px.index >= start) & (px.index <= end)]
sec_px_window = restrict_window(sec_px, START_DATE, END_DATE)

def filter_eligible_securities(px: pd.DataFrame) -> pd.DataFrame:
    start_prices = px.iloc[0]
    end_prices   = px.iloc[-1]

    eligible = start_prices.notna() & end_prices.notna()
    px = px.loc[:, eligible]

    returns = px.pct_change()
    has_returns = returns.notna().sum() > 0

    return px.loc[:, has_returns]
sec_px_clean = filter_eligible_securities(sec_px_window)

In [ ]:
def load_sec_metadata(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path)
    meta.columns = meta.columns.str.lower()
    meta["date"] = pd.to_datetime(meta["date"])
    return meta

sec_metadata = load_sec_metadata("sec_meta_data.csv")

def build_name_map(meta: pd.DataFrame) -> pd.Series:
    latest = (
        meta.sort_values("date")
            .groupby("sedol")
            .tail(1)
    )
    return latest.set_index("sedol")["name"]
name_map = build_name_map(sec_metadata)

In [ ]:
print(sec_px.index.min(), sec_px.index.max())
print(sec_metadata["date"].min(), sec_metadata["date"].max())